In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.tree import DecisionTreeClassifier

# (a) Data Loading and Splitting

In [8]:
import pandas as pd

digits = load_digits()
X, y = digits.data, digits.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples\n")

# Create a DataFrame for visualization
df = pd.DataFrame(data=X, columns=[f'pixel_{i}' for i in range(X.shape[1])])
df['target'] = y
print(df.head())

Training set: 1347 samples
Test set: 450 samples

   pixel_0  pixel_1  pixel_2  pixel_3  pixel_4  pixel_5  pixel_6  pixel_7  \
0      0.0      0.0      5.0     13.0      9.0      1.0      0.0      0.0   
1      0.0      0.0      0.0     12.0     13.0      5.0      0.0      0.0   
2      0.0      0.0      0.0      4.0     15.0     12.0      0.0      0.0   
3      0.0      0.0      7.0     15.0     13.0      1.0      0.0      0.0   
4      0.0      0.0      0.0      1.0     11.0      0.0      0.0      0.0   

   pixel_8  pixel_9  ...  pixel_55  pixel_56  pixel_57  pixel_58  pixel_59  \
0      0.0      0.0  ...       0.0       0.0       0.0       6.0      13.0   
1      0.0      0.0  ...       0.0       0.0       0.0       0.0      11.0   
2      0.0      0.0  ...       0.0       0.0       0.0       0.0       3.0   
3      0.0      8.0  ...       0.0       0.0       0.0       7.0      13.0   
4      0.0      0.0  ...       0.0       0.0       0.0       0.0       2.0   

   pixel_60  pixel

# Impurity Functions

In [3]:
def calculate_entropy(y):
    if len(y) == 0: return 0
    _, counts = np.unique(y, return_counts=True)
    probabilities = counts / len(y)
    return -np.sum(probabilities * np.log2(probabilities + 1e-9))

def calculate_gini(y):
    if len(y) == 0: return 0
    _, counts = np.unique(y, return_counts=True)
    probabilities = counts / len(y)
    return 1.0 - np.sum(probabilities ** 2)

# (b) Custom Decision Tree Classifier

In [5]:
class SimpleDecisionTree:
    def __init__(self, criterion='gini', max_depth=10):
        self.criterion = criterion
        self.max_depth = max_depth
        self.tree = None
        self.root_impurity = None
        self.root_gain = None

    def fit(self, X, y):
        # Record root node impurity before any splits
        self.root_impurity = calculate_entropy(y) if self.criterion == 'entropy' else calculate_gini(y)
        self.tree = self._build_tree(X, y, depth=0)

    def _build_tree(self, X, y, depth):
        num_samples, num_features = X.shape
        unique_classes = np.unique(y)

        # Stopping criteria
        if len(unique_classes) == 1 or depth >= self.max_depth:
            most_common_label = unique_classes[np.argmax(np.unique(y, return_counts=True)[1])]
            return {'value': most_common_label}

        best_feat, best_thresh, best_gain = self._best_split(X, y, num_features)

        # If no valid split is found (gain is 0)
        if best_gain == 0:
            most_common_label = unique_classes[np.argmax(np.unique(y, return_counts=True)[1])]
            return {'value': most_common_label}

        if depth == 0:
            self.root_gain = best_gain

        # Split the data
        left_idx = X[:, best_feat] <= best_thresh
        right_idx = X[:, best_feat] > best_thresh

        # Recursively build left and right branches
        left_branch = self._build_tree(X[left_idx, :], y[left_idx], depth + 1)
        right_branch = self._build_tree(X[right_idx, :], y[right_idx], depth + 1)

        return {'feature': best_feat, 'threshold': best_thresh,
                'left': left_branch, 'right': right_branch}

    def _best_split(self, X, y, num_features):
        best_gain = -1
        best_feat, best_thresh = None, None
        current_impurity = calculate_entropy(y) if self.criterion == 'entropy' else calculate_gini(y)

        for feat_idx in range(num_features):
            thresholds = np.unique(X[:, feat_idx])
            for thresh in thresholds:
                left_y = y[X[:, feat_idx] <= thresh]
                right_y = y[X[:, feat_idx] > thresh]

                if len(left_y) == 0 or len(right_y) == 0:
                    continue

                # Calculate weighted impurity of children
                n = len(y)
                if self.criterion == 'entropy':
                    child_impurity = (len(left_y)/n) * calculate_entropy(left_y) + (len(right_y)/n) * calculate_entropy(right_y)
                else:
                    child_impurity = (len(left_y)/n) * calculate_gini(left_y) + (len(right_y)/n) * calculate_gini(right_y)

                # Information Gain
                gain = current_impurity - child_impurity

                if gain > best_gain:
                    best_gain = gain
                    best_feat = feat_idx
                    best_thresh = thresh

        return best_feat, best_thresh, best_gain

    def predict(self, X):
        return np.array([self._traverse_tree(x, self.tree) for x in X])

    def _traverse_tree(self, x, node):
        if 'value' in node:
            return node['value']
        if x[node['feature']] <= node['threshold']:
            return self._traverse_tree(x, node['left'])
        return self._traverse_tree(x, node['right'])

# Train Custom Models
custom_tree_entropy = SimpleDecisionTree(criterion='entropy', max_depth=10)
custom_tree_entropy.fit(X_train, y_train)
y_pred_custom_ent = custom_tree_entropy.predict(X_test)

custom_tree_gini = SimpleDecisionTree(criterion='gini', max_depth=10)
custom_tree_gini.fit(X_train, y_train)
y_pred_custom_gini = custom_tree_gini.predict(X_test)

# (c) Scikit-Learn Decision Tree Classifier

In [6]:
sk_tree_entropy = DecisionTreeClassifier(criterion='entropy', max_depth=10, random_state=42)
sk_tree_entropy.fit(X_train, y_train)
y_pred_sk_ent = sk_tree_entropy.predict(X_test)

sk_tree_gini = DecisionTreeClassifier(criterion='gini', max_depth=10, random_state=42)
sk_tree_gini.fit(X_train, y_train)
y_pred_sk_gini = sk_tree_gini.predict(X_test)

# Reporting Requirements

In [7]:
print("\nRoot Node Impurity & Gain Reports")
print("1. Using Information Gain (Entropy)")
print(f"   Custom Model Root Entropy: {custom_tree_entropy.root_impurity:.4f}")
print(f"   Scikit-Learn Root Entropy: {sk_tree_entropy.tree_.impurity[0]:.4f}")

print("\n2. Using Gini Index")
print(f"   Custom Model Root Gini:    {custom_tree_gini.root_impurity:.4f}")
print(f"   Scikit-Learn Root Gini:    {sk_tree_gini.tree_.impurity[0]:.4f}")


print("\nTest Data Accuracies")
print("1. Information Gain (Entropy)")
print(f"   Custom Model Accuracy: {accuracy_score(y_test, y_pred_custom_ent):.4f}")
print(f"   Scikit-Learn Accuracy: {accuracy_score(y_test, y_pred_sk_ent):.4f}")

print("\n2. Gini Index")
print(f"   Custom Model Accuracy: {accuracy_score(y_test, y_pred_custom_gini):.4f}")
print(f"   Scikit-Learn Accuracy: {accuracy_score(y_test, y_pred_sk_gini):.4f}")

# Extract sample labels
print("\nSample Labels Generated on Test Data (First 15)")
print(f"True Labels:      {y_test[:15]}")
print(f"Custom (Entropy): {y_pred_custom_ent[:15]}")
print(f"Sklearn (Entropy):{y_pred_sk_ent[:15]}")



Root Node Impurity & Gain Reports
1. Using Information Gain (Entropy)
   Custom Model Root Entropy: 3.3205
   Scikit-Learn Root Entropy: 3.3205

2. Using Gini Index
   Custom Model Root Gini:    0.8998
   Scikit-Learn Root Gini:    0.8998

Test Data Accuracies
1. Information Gain (Entropy)
   Custom Model Accuracy: 0.8711
   Scikit-Learn Accuracy: 0.8756

2. Gini Index
   Custom Model Accuracy: 0.8733
   Scikit-Learn Accuracy: 0.8667

Sample Labels Generated on Test Data (First 15)
True Labels:      [6 9 3 7 2 1 5 2 5 2 1 9 4 0 4]
Custom (Entropy): [6 9 3 7 2 1 5 3 5 7 1 9 0 0 4]
Sklearn (Entropy):[6 9 3 7 2 1 5 3 5 7 1 9 6 0 4]


# Plotting Confusion Matrices

In [10]:
# fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# cm_ent = confusion_matrix(y_test, y_pred_custom_ent)
# disp_ent = ConfusionMatrixDisplay(confusion_matrix=cm_ent)
# disp_ent.plot(ax=axes[0], cmap='Blues', colorbar=False)
# axes[0].set_title("Custom Tree (Information Gain)")

# cm_gini = confusion_matrix(y_test, y_pred_custom_gini)
# disp_gini = ConfusionMatrixDisplay(confusion_matrix=cm_gini)
# disp_gini.plot(ax=axes[1], cmap='Greens', colorbar=False)
# axes[1].set_title("Custom Tree (Gini Index)")

# plt.tight_layout()
# plt.show()